In [1]:
from unsloth import FastLanguageModel
import transformers
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import argparse
import torch
import numpy as np
import torch.nn.functional as F
from torch.distributions import Categorical
import gc
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [52]:
MODEL   = "marcelbinz/Llama-3.1-Centaur-70B-adapter" # marcelbinz/Llama-3.1-Centaur-70B-adapter   marcelbinz/Llama-3.1-Minitaur-8B-adapter
DATA    = "Data/narrative_data.csv"
DOMAIN  = "Mammals"
TESTING = True

In [23]:
# Check if GPU works
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, flush=True)

cuda


In [24]:
# Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL,
        max_seq_length = 32768, # Reduced for memory efficiency
        dtype          = None,
        load_in_4bit   = True)

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 2. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-70B-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/414M [00:00<?, ?B/s]

Unsloth 2026.4.8 patched 80 layers with 80 QKV layers, 80 O layers and 80 MLP layers.


In [25]:
%%capture
FastLanguageModel.for_inference(model)

In [26]:
l_id = tokenizer(" <<").input_ids[1:]
r_id = tokenizer(">>").input_ids[1:]
print(l_id, flush=True)
print(r_id, flush=True)

[1134]
[2511]


In [27]:
# The function's job is: take a batch of tokenized sequences, and return a labels tensor 
# where only the response tokens (between << and >>) are unmasked.
# Everything else gets -100 so PyTorch ignores it in the loss/NLL calculation.

IGNORE_INDEX = -100

def collate_fn(batch):
    input_ids_list      = [x["input_ids"] for x in batch]
    attention_mask_list = [x["attention_mask"] for x in batch]
    participants        = [x["participant"] for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask_list, batch_first=True, padding_value=0
    )

    labels = torch.full_like(input_ids, IGNORE_INDEX)  # start: everything masked

    l_id_list = list(l_id)  # tokens for " <<"
    r_id_list = list(r_id)  # tokens for ">>"

    for i in range(input_ids.size(0)):
        seq = input_ids[i].tolist()
        j = 0
        while j < len(seq):
            # Look for << (response start)
            if seq[j:j+len(l_id_list)] == l_id_list:
                response_start = j + len(l_id_list)  # first token AFTER 
                # Now look for >> (response end)
                k = response_start
                while k < len(seq):
                    if seq[k:k+len(r_id_list)] == r_id_list:
                        response_end = k  # token AT >>
                        # Unmask the response tokens (between << and >>)
                        labels[i, response_start:response_end] = input_ids[i, response_start:response_end]
                        j = response_end + len(r_id_list)
                        break
                    k += 1
                else:
                    j += 1
            else:
                j += 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "participant": torch.tensor(participants),
    }

In [58]:
dataset =  load_dataset("csv", data_files=DATA)["train"]

In [59]:
dataset = dataset.filter(lambda x: x["domain"] == DOMAIN)

In [60]:
dataset.to_pandas().head(2)

,text,domain,participant,ID
0,Your task is to estimate the days until female...,Mammals,0,0xhc0yxm5czl
1,Your task is to estimate the days until female...,Mammals,1,2qewr80masfh


In [61]:
def tokenization(example):
      print(example)
      tokenized = tokenizer(example['text'])
      tokenized['participant'] = int(example['participant'])
      return tokenized

In [62]:
%%capture
dataset = dataset.map(tokenization).sort('participant')
# print(dataset)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "participant"])

In [63]:
dataloader = torch.utils.data.DataLoader(
    dataset    = dataset,
    collate_fn = collate_fn,
    batch_size = 1
)

In [64]:
nlls = []
with torch.no_grad():
    for data_part in tqdm(dataloader, desc="Participants"):
        participant = data_part["participant"].item()  
        
        model_outputs = model(data_part['input_ids'].to(device), data_part['attention_mask'].to(device), return_dict=True)
        targets_ids = data_part['labels'][0, 1:].detach().cpu()

        nll = torch.nn.functional.cross_entropy(model_outputs.logits[0, :-1].detach().cpu(), targets_ids, reduction='none')
        nll = nll[targets_ids != -100]
        nlls.append(nll)

        del targets_ids
        del model_outputs
        torch.cuda.empty_cache()
        gc.collect()

Participants: 100%|██████████| 48/48 [03:25<00:00,  4.28s/it]


In [65]:
torch.save(nlls, 'Results/log_likelihood_' + DOMAIN + "_"+ MODEL.replace('/', '-') +  '.pth')